# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

i will choose two signals : staleness and ctr-vs-position
so my simple rule is : the page needs to refresh if its ctr is low compared to its position and it's old


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os
df = pd.read_csv('/content/content_refresh_anonymized.csv')

stale = df['freshness_tier'].isin(['91-180','181+']).astype(int)
visible = (df['impressions_last_30d']>=10).astype(int)
tier_median_ctr = df[df['impressions_last_30d']>=10].groupby('position_tier')['ctr'].median()
expected_ctr = df['position_tier'].map(tier_median_ctr)
low_ctr = (df["ctr"] < expected_ctr).astype(int)
df['score'] = stale * visible * low_ctr * df['impressions_last_30d']
df["reason_code"] = "not_flagged"
df.loc[df["score"] > 0, "reason_code"] = "stale_low_ctr_for_position"

# --- Action label ---
df["action"] = "no_action"
df.loc[df["score"] > 0, "action"] = "refresh"

# --- Rank everything and write the CSV ---
os.makedirs("work/outputs", exist_ok=True)

out_cols = ["content_id", "client_id", "score", "reason_code", "action",
            "freshness_tier", "position_tier", "ctr", "impressions_last_30d"]

ranked = df[out_cols].sort_values("score", ascending=False).reset_index(drop=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("rows written:", len(ranked))
ranked.head(10)

rows written: 30000


,content_id,client_id,score,reason_code,action,freshness_tier,position_tier,ctr,impressions_last_30d
0,content_4a6607efcb46,client_6208ef0f77,122303,stale_low_ctr_for_position,refresh,91-180,top_3,0.01,122303
1,content_5fe46e04994d,client_4e07408562,120791,stale_low_ctr_for_position,refresh,91-180,page_1,0.14,120791
2,content_36ff89c8214e,client_19581e27de,106985,stale_low_ctr_for_position,refresh,91-180,page_1,0.05,106985
3,content_91652435f57a,client_19581e27de,74135,stale_low_ctr_for_position,refresh,91-180,page_1,0.06,74135
4,content_cb112fce36be,client_19581e27de,72468,stale_low_ctr_for_position,refresh,91-180,page_1,0.16,72468
5,content_a7427266c305,client_19581e27de,63346,stale_low_ctr_for_position,refresh,91-180,page_1,0.11,63346
6,content_c8e9d6ab9013,client_19581e27de,63326,stale_low_ctr_for_position,refresh,91-180,page_1,0.00,63326
7,content_33b4dceecad1,client_19581e27de,59344,stale_low_ctr_for_position,refresh,91-180,page_1,0.16,59344
8,content_4c76e9b13aea,client_19581e27de,55948,stale_low_ctr_for_position,refresh,91-180,page_1,0.07,55948
9,content_b115f7c74779,client_19581e27de,51115,stale_low_ctr_for_position,refresh,91-180,page_1,0.03,51115


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_20 = ranked.head(20)
print(top_20)

              content_id          client_id   score  \
0   content_4a6607efcb46  client_6208ef0f77  122303   
1   content_5fe46e04994d  client_4e07408562  120791   
2   content_36ff89c8214e  client_19581e27de  106985   
3   content_91652435f57a  client_19581e27de   74135   
4   content_cb112fce36be  client_19581e27de   72468   
5   content_a7427266c305  client_19581e27de   63346   
6   content_c8e9d6ab9013  client_19581e27de   63326   
7   content_33b4dceecad1  client_19581e27de   59344   
8   content_4c76e9b13aea  client_19581e27de   55948   
9   content_b115f7c74779  client_19581e27de   51115   
10  content_f42eb861c6dd  client_19581e27de   47350   
11  content_11fcfd65d94c  client_19581e27de   44335   
12  content_5d3dfb80a423  client_19581e27de   43905   
13  content_154aa47edd03  client_19581e27de   42241   
14  content_c1fe78bc4e37  client_19581e27de   38658   
15  content_45fb95832c96  client_19581e27de   34994   
16  content_d312eb371bcf  client_19581e27de   33703   
17  conten

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

this rule has something wrong, when we check if top 10 pages of our calculations by checking if that pages is really down from trend_direction we will find that only 3/10 is true and the rest is false which gives us 30% accuracy of that rule, in addition if we took a random 10 pages the percentage is 54% which is bad but at least better than ours

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
columns_used_in_score = ["freshness_tier", "impressions_last_30d", "position_tier", "ctr"]
leaked = [c for c in ["trend_direction", "trend_pct"] if c in columns_used_in_score]
print("Leaked columns:", leaked if leaked else "None ")

Leaked columns: None 


In [21]:
top_20 = df.sort_values("score", ascending=False).head(20)
top_20[["content_id", "score", "reason_code", "action", "trend_direction", "freshness_tier", "position_tier", "ctr"]]

,content_id,score,reason_code,action,trend_direction,freshness_tier,position_tier,ctr
3331,content_4a6607efcb46,122303,stale_low_ctr_for_position,refresh,up,91-180,top_3,0.01
6653,content_5fe46e04994d,120791,stale_low_ctr_for_position,refresh,down,91-180,page_1,0.14
3394,content_36ff89c8214e,106985,stale_low_ctr_for_position,refresh,stable,91-180,page_1,0.05
3070,content_91652435f57a,74135,stale_low_ctr_for_position,refresh,stable,91-180,page_1,0.06
26531,content_cb112fce36be,72468,stale_low_ctr_for_position,refresh,down,91-180,page_1,0.16
26474,content_a7427266c305,63346,stale_low_ctr_for_position,refresh,stable,91-180,page_1,0.11
7445,content_c8e9d6ab9013,63326,stale_low_ctr_for_position,refresh,down,91-180,page_1,0.00
18099,content_33b4dceecad1,59344,stale_low_ctr_for_position,refresh,stable,91-180,page_1,0.16
2476,content_4c76e9b13aea,55948,stale_low_ctr_for_position,refresh,up,91-180,page_1,0.07
4708,content_b115f7c74779,51115,stale_low_ctr_for_position,refresh,up,91-180,page_1,0.03


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.